In [ ]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
import random


def random_flip(image):
    if random.choice([True, False]):
        image = np.fliplr(image)
    return image


def add_salt_and_pepper_noise(image, salt_prob=0.01, pepper_prob=0.01):
    noisy_image = image.copy()
    total_pixels = image.size

    num_salt = int(salt_prob * total_pixels)
    salt_coords = [np.random.randint(0, i - 1, num_salt) for i in image.shape]
    noisy_image[salt_coords[0], salt_coords[1]] = 255

    num_pepper = int(pepper_prob * total_pixels)
    pepper_coords = [np.random.randint(0, i - 1, num_pepper) for i in image.shape]
    noisy_image[pepper_coords[0], pepper_coords[1]] = 0

    return noisy_image


def normalize_depth(depth_img):
    depth_min = np.min(depth_img)
    depth_max = np.max(depth_img)
    normalized_depth = (depth_img - depth_min) / (depth_max - depth_min) * 255
    return normalized_depth.astype(np.uint8)


def enhance_depth_contrast(depth_img, gamma=2.0):
    normalized_depth = normalize_depth(depth_img)
    enhanced_depth = np.power(normalized_depth / 255.0, gamma) * 255
    return np.clip(enhanced_depth, 0, 255).astype(np.uint8)


def resize_w_enhanced_depth(image, target_size=(256, 384), gamma=1.2):
    enhanced_depth_image = enhance_depth_contrast(image, gamma)
    resized_image = cv.resize(
        enhanced_depth_image, target_size, interpolation=cv.INTER_LINEAR
    )
    return resized_image

In [ ]:
total = {}
AUGMENT = False
for batch in os.listdir("batches"):
    batch_path = os.path.join("batches", batch)
    if os.path.isdir(batch_path):
        df = []
        for item in os.listdir(batch_path):
            item_path = os.path.join(batch_path, item)
            if os.path.isdir(item_path):
                for img in os.listdir(item_path):
                    img_path = os.path.join(item_path, img)
                    if os.path.isfile(img_path) and "depth" in img_path:
                        image = Image.open(img_path)
                        image = np.array(image)
                        image[image <= 1] = 0
                        df.append(image)
                        if AUGMENT:
                            for _ in range(4):
                                df.append(random_flip(image))

                            for deg in range(random.randint(a=1, b=5)):
                                peppered = add_salt_and_pepper_noise(
                                    image, salt_prob=deg * 0.02, pepper_prob=deg * 0.02
                                )
                                df.append(peppered)

                                for _ in range(4):
                                    df.append(random_flip(peppered))

                            df.append(add_salt_and_pepper_noise(image))

            total[item_path] = {"data": df}

In [ ]:
reference = 1280 * 480
for key, df in total.items():
    pixel_sizes = [x.shape[0] * x.shape[1] for x in df["data"]]
    total[key]["pixel_sizes"] = np.array(pixel_sizes)
    median_depth = [np.median(x) for x in df["data"]]
    total[key]["median_depth"] = np.array(median_depth)
    min_depth = [np.min(x) for x in df["data"]]
    max_depth = [np.max(x) for x in df["data"]]
    total[key]["min_depth"] = np.array(min_depth)
    total[key]["max_depth"] = np.array(max_depth)
    depth_range = [mx - mi for mx, mi in zip(max_depth, min_depth)]
    total[key]["depth_range"] = np.array(depth_range)
    std_depth = [np.std(x) for x in df["data"]]
    total[key]["std_depth"] = np.array(std_depth)
    pixel_ratio = np.array([size / reference for size in pixel_sizes])
    total[key]["pixel_ratio"] = pixel_ratio
    adjusted_df = [resize_w_enhanced_depth(x) for x in df["data"]]
    total[key]["data"] = adjusted_df

In [ ]:
for (key, df), w in zip(total.items(), [15.21828, 17.3282571, 17.0191098, 16.7612445]):
    total[key]["weight"] = w

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch.optim as optim


class DepthImageDataset(Dataset):
    def __init__(self, data_dict):
        self.images = []
        self.features = []
        self.weights = []

        for key in data_dict.keys():
            images = data_dict[key]["data"]
            features = np.array(
                [
                    data_dict[key]["median_depth"],
                    data_dict[key]["min_depth"],
                    data_dict[key]["max_depth"],
                    data_dict[key]["depth_range"],
                    data_dict[key]["std_depth"],
                    data_dict[key]["pixel_sizes"],
                    data_dict[key]["pixel_ratio"],
                ]
            ).T
            weights = [data_dict[key]["weight"] for x in range(len(images))]
            self.images.extend(images)
            self.features.extend(features)
            self.weights.extend(weights)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        feature = torch.tensor(self.features[idx], dtype=torch.float32)
        weight = torch.tensor(self.weights[idx], dtype=torch.float32)
        return image, feature, weight

In [ ]:
#Change the number of keys to fit what you have. In this example, 2 pigs will be used to train, and the last 2 will be used for validation.
train_keys = list(total.keys())[:-3]
val_keys = list(total.keys())[:-2]
train_data_dict = {key: total[key] for key in train_keys}
val_data_dict = {key: total[key] for key in val_keys}

In [ ]:
train_dataset = DepthImageDataset(train_data_dict)
val_dataset = DepthImageDataset(val_data_dict)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
class CNN(nn.Module):
    def __init__(self, feature_input_size):
        super(CNN, self).__init__()
        # You can not change the conv1(in_channels) and fc4 (out_features).
        # Do not change kernel_size, stride, padding.
        # You can change the out channels but make sure to adjust for the consequent layer.
        # Example: conv1(out_channels) is 16 so conv2(in_channels) must be 16 or conv1(out_channels) = conv2(in_channels).
        self.conv1 = nn.Conv2d(
            in_channels=4, out_channels=16, kernel_size=3, stride=1, padding=1
        )
        self.conv2 = nn.Conv2d(
            in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1
        )
        self.conv3 = nn.Conv2d(
            in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(64 * 32 * 48, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(feature_input_size, 128)
        self.fc4 = nn.Linear(256 + 128, out_features=1)

    def forward(self, x_image, x_features):
        x_image = self.pool(torch.relu(self.conv1(x_image)))
        x_image = self.pool(torch.relu(self.conv2(x_image)))
        x_image = self.pool(torch.relu(self.conv3(x_image)))

        x_image = x_image.reshape(x_image.size(0), -1)
        x_image = torch.relu(self.fc1(x_image))
        x_image = torch.relu(self.fc2(x_image))
        
        x_features = torch.relu(self.fc3(x_features))
        
        x = torch.cat((x_image, x_features), dim=1)
        x = self.fc4(x)
        return x

In [ ]:
#If you want to add or remove features you must also change the size of this.

model = CNN(feature_input_size=7)
model = model.to(device="cuda")

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for images, features, weights in train_dataloader:
        images = images.to(device="cuda")
        features = features.to(device="cuda")
        weights = weights.to(device="cuda")

        optimizer.zero_grad()
        
        #Do not forget to permute the images because the in_channels is 4 (the last column of our image.)
        outputs = model(images.permute(0, 3, 1, 2), features)

        loss = criterion(outputs, weights)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(train_dataloader)}"
    )

In [ ]:
model.eval()
device = "cuda"

val_rmse = 0
num_batches = 0

with torch.no_grad():
    for images, features, labels in val_dataloader:
        images, features, labels = (
            images.to(device),
            features.to(device),
            labels.to(device),
        )
        outputs = model(images.permute(0, 3, 1, 2), features)
        mse = nn.MSELoss()(outputs, labels.unsqueeze(-1))
        rmse = torch.sqrt(mse)
        val_rmse += rmse.item()
        num_batches += 1

avg_val_rmse = val_rmse / num_batches

print(f"Validation RMSE: {avg_val_rmse}")